# Example 6.2.1 / 2: Conditions Based on Solver Values

Companion notebook for **6.2.1 repeat Loops, break and continue** in the AMPL lab Documentation, section "Conditions Based on Solver Values". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Conditions Based on Solver Values

Solver results use finite-precision arithmetic. Avoid termination tests such as
`until Capacity.dual = 0` unless the values have deliberately been rounded. A tolerance test,
such as `until abs(Capacity.dual) <= 1e-7`, usually expresses the operational intent more safely.
Always impose a separate iteration limit when convergence is not mathematically guaranteed. This
loop raises the capacity by 10 after each solve until the profit reaches 100, stops early on a
bad solve, and caps the loop at 20 passes:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param capacity >= 0 default 10;          # available capacity, raised after each solve
var Make >= 0;                           # units made
maximize Profit: 4 * Make;               # profit of 4 per unit
subject to Capacity: Make <= capacity;   # production cannot exceed capacity
param passes integer default 0;          # solves done so far

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.option["solver"] = "highs"
ampl.option["solver_msg"] = 0
ampl.eval("""
repeat {
    solve;
    let passes := passes + 1;
    if solve_result <> "solved" then break;
    if Profit >= 100 then break;
    let capacity := capacity + 10;
} until passes >= 20;
""")
print(ampl.param["passes"].value(), ampl.param["capacity"].value(), ampl.obj["Profit"].value())

3 30 120.0


### Pure AMPL: model.run

The same in pure AMPL, with the same `model.mod`:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
repeat {
    solve;
    let passes := passes + 1;
    if solve_result <> "solved" then break;
    if Profit >= 100 then break;
    let capacity := capacity + 10;
} until passes >= 20;
display passes, capacity, Profit;

passes = 3
capacity = 30
Profit = 120

